# Task 04 — Supervised Modeling & Hyperparameter Tuning
**LATHIKA I (25AD311)**

Four classification models are tuned and compared using stratified cross-validation. Dataset is synthetic; metrics are for learning only.

In [ ]:
import pandas as pd, numpy as np, joblib
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, RocCurveDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
p=Path('../data/synthetic_fraud_classification.csv')
if not p.exists(): p=Path('synthetic_fraud_classification.csv')
df=pd.read_csv(p)
df.head()

In [ ]:
X=df.drop(columns=['fraud_flag','record_id']); y=df.fraud_flag
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,random_state=404,stratify=y)
num=X.select_dtypes(include='number').columns.tolist(); cat=X.select_dtypes(exclude='number').columns.tolist()
pre=ColumnTransformer([('num',Pipeline([('imp',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num),('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('ohe',OneHotEncoder(handle_unknown='ignore'))]),cat)])
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=404)

In [ ]:
model_grids={
 'Logistic Regression':(LogisticRegression(max_iter=1500,class_weight='balanced'),{'model__C':[.1,1,10]}),
 'Decision Tree':(DecisionTreeClassifier(class_weight='balanced',random_state=404),{'model__max_depth':[3,6,None],'model__min_samples_leaf':[2,5]}),
 'Random Forest':(RandomForestClassifier(class_weight='balanced',random_state=404,n_jobs=-1),{'model__n_estimators':[100,200],'model__max_depth':[None,8],'model__min_samples_leaf':[1,3]}),
 'SVC':(SVC(probability=True,class_weight='balanced',random_state=404),{'model__C':[.5,1,2],'model__kernel':['rbf','linear']})}
results=[]; fitted_models={}; roc_data={}
for name,(est,grid) in model_grids.items():
 pipe=Pipeline([('prep',pre),('model',est)])
 search=GridSearchCV(pipe,grid,scoring='roc_auc',cv=cv,n_jobs=-1,refit=True)
 search.fit(X_train,y_train)
 fitted=search.best_estimator_; fitted_models[name]=fitted
 pred=fitted.predict(X_test); prob=fitted.predict_proba(X_test)[:,1]
 results.append({'model':name,'best_cv_roc_auc':search.best_score_,'test_precision':precision_score(y_test,pred,zero_division=0),'test_recall':recall_score(y_test,pred,zero_division=0),'test_f1':f1_score(y_test,pred,zero_division=0),'test_roc_auc':roc_auc_score(y_test,prob),'best_params':search.best_params_})
 roc_data[name]=prob
 print(name,search.best_params_,classification_report(y_test,pred,zero_division=0))
comparison=pd.DataFrame(results).sort_values('best_cv_roc_auc',ascending=False)
comparison

In [ ]:
champion_name=comparison.iloc[0]['model']
champion=fitted_models[champion_name]
print('Champion by CV ROC-AUC:',champion_name)
champ_pred=champion.predict(X_test)
print('Confusion matrix:\n',confusion_matrix(y_test,champ_pred))

In [ ]:
plt.figure(figsize=(7,5))
ax=plt.gca()
for name,prob in roc_data.items(): RocCurveDisplay.from_predictions(y_test,prob,name=name,ax=ax)
plt.title('ROC Curves — Synthetic Test Set'); plt.tight_layout(); plt.show()

In [ ]:
# Save champion pipeline; includes preprocessing and classifier
joblib.dump(champion,'champion_model.joblib')
print('Saved champion_model.joblib')

## Responsible interpretation
The model and scores use artificial data. Do not use this model for actual fraud decisions. For real data, validate provenance, leakage, calibration, subgroup errors, operational costs, human review, monitoring and rollback.